# Transformer Task

[video](https://www.youtube.com/watch?v=U0s0f995w14#:~:text=In%20this%20video%20we%20read%20the%20original,http://www.peterbloem.nl/blog/transformers%20%E2%9D%A4%EF%B8%8F%20Support%20the%20channel%20%E2%9D%A4%EF%B8%8F%20https://www.youtube.com/)

## Code

In [ ]:
import math

import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn
import models.deep_learning.components as comp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

# Encoding

In [3]:
class InputEmbedding(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.word_embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.word_embedding(x) * math.sqrt(self.d_model)


# Projection Layer

In [4]:
class Projection(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.head(x), dim=-1)

## Hyper-Parameters

In [5]:
class Translator(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        src_max_length: int = 100,
        tgt_max_length: int = 100,
        embed_size: int = 256,
        num_layers: int = 6,
        heads: int = 8,
        forward_dimension: int = 2048,
        dropout: float = 0.0,
        device: torch.device | None = None,
    ):
        super().__init__()

        # Encoder
        self.src_embedding = nn.Sequential(
            InputEmbedding(d_model=embed_size, vocab_size=src_vocab_size),
            comp.SinusoidalPE(d_model=embed_size, max_len=src_max_length),
            nn.Dropout(0.1),
        )
        self.encoder_layer = mynn.TransformerEncoderLayer(
            d_model=embed_size,
            nhead=heads,
            dim_feedforward=forward_dimension,
            dropout=dropout,
            device=device,
        )
        self.encoder = mynn.TransformerEncoder(
            encoder_layer=self.encoder_layer,
            num_layers=num_layers,
        )

        # Decoder
        self.tgt_embedding = nn.Sequential(
            InputEmbedding(d_model=embed_size, vocab_size=tgt_vocab_size),
            comp.SinusoidalPE(d_model=embed_size, max_len=tgt_max_length),
            nn.Dropout(0.1),
        )
        self.decoder_layer = mynn.TransformerDecoderLayer(
            d_model=embed_size,
            nhead=heads,
            dim_feedforward=forward_dimension,
            dropout=dropout,
            device=device,
        )
        self.decoder = mynn.TransformerDecoder(
            decoder_layer=self.decoder_layer,
            num_layers=num_layers,
        )

        # Final linear layer
        self.projection = Projection(d_model=embed_size, vocab_size=tgt_vocab_size)
        self.device = device

    # def make_src_mask(self, src: torch.Tensor) -> torch.Tensor:
    #     # src: (N, src_len)
    #     src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
    #     # src_mask: (N, 1, 1, src_len)
    #     return src_mask.to(self.device)

    # def make_tgt_mask(self, tgt: torch.Tensor) -> torch.Tensor:
    #     # tgt: (N, tgt_len)
    #     N, tgt_len = tgt.shape
    #     tgt_mask = torch.tril(
    #         torch.ones((tgt_len, tgt_len), device=self.device)
    #     ).expand(N, 1, tgt_len, tgt_len)
    #     # tgt_mask: (N, 1, tgt_len, tgt_len)
    #     return tgt_mask

    def encode(self, src: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
        return self.encoder(self.src_embedding(src), mask=src_mask)

    def decode(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor,
        src_mask: torch.Tensor,
    ) -> torch.Tensor:
        return self.decoder(
            self.tgt_embedding(tgt),
            memory,
            tgt_mask=tgt_mask,
            memory_mask=src_mask,
        )

    def project(self, dec_output: torch.Tensor) -> torch.Tensor:
        return self.projection(dec_output)

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,
        src_mask: torch.Tensor,
        tgt_mask: torch.Tensor,
    ) -> torch.Tensor:
        # src_mask = self.make_src_mask(src)
        # tgt_mask = self.make_tgt_mask(tgt)

        enc_src = self.encode(src, src_mask)
        dec_output = self.decode(tgt, enc_src, tgt_mask, src_mask)
        output = self.project(dec_output)

        return output

### Example

In [6]:
x = torch.tensor([[1, 5, 6, 4, 3, 9, 5, 2, 0], [1, 8, 7, 3, 4, 5, 6, 7, 2]]).to(device)

trg = torch.tensor([[1, 7, 4, 3, 5, 9, 2, 0], [1, 5, 6, 2, 4, 7, 6, 2]]).to(device)

src_pad_idx = 0
trg_pad_idx = 0
src_vocab_size = 10
trg_vocab_size = 10

translator = Translator(src_vocab_size, trg_vocab_size).to(device)
for param in translator.parameters():
    if param.dim() > 1:
        nn.init.xavier_uniform_(param)

out = translator(x, trg[:, :-1], None, None)

print(out.shape)

torch.Size([2, 7, 10])


## Dataset

## Testing

In [77]:
src_vocab_size = 1000
embed_size = 512
num_layers = 6
heads = 8
forward_expansion = 4
max_length = 100

In [78]:
# input parameters
N = 3
batch_size = 2
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float32

# Encoder Layer parameters
dropout = 0.2
activation = nn.GELU
layer_norm_eps = 1e-5
batch_first = True
norm_first = True
bias = True

## Transformer encoder parameters
num_layers = 3
#  It helps only when norm_first is True,
norm = None  # nn.LayerNorm(d_model).to(device=device,dtype=dtype)

In [79]:
tf_encl = mynn.TransformerEncoderLayer(
    d_model=embed_size,
    nhead=heads,  # Assuming each head has 64 dimensions
    dim_feedforward=embed_size * forward_expansion,
    dropout=dropout,
    activation_cls=activation,
    layer_norm_eps=layer_norm_eps,
    batch_first=batch_first,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)
tf_enc = mynn.TransformerEncoder(tf_encl, num_layers, norm=norm)


TypeError: TransformerEncoderLayer.__init__() got an unexpected keyword argument 'batch_first'

### Evaluation

In [ ]:
tf_enc.eval()
tf_enc(x)

tensor([[[-0.0296, -0.0916,  1.0885, -1.0260],
         [ 0.6635,  0.4575,  1.4182, -0.2418],
         [ 0.4502,  0.3450,  0.7149,  0.3737]],

        [[ 0.1837, -0.0224,  1.0295,  1.6342],
         [ 0.3983, -0.1241,  0.9514,  0.5204],
         [ 1.1837,  1.0843,  1.0421,  0.6918]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [ ]:
nn_tf_enc.eval()
nn_tf_enc(x)

tensor([[[-0.0296, -0.0916,  1.0885, -1.0260],
         [ 0.6635,  0.4575,  1.4182, -0.2418],
         [ 0.4502,  0.3450,  0.7149,  0.3737]],

        [[ 0.1837, -0.0224,  1.0295,  1.6342],
         [ 0.3983, -0.1241,  0.9514,  0.5204],
         [ 1.1837,  1.0843,  1.0421,  0.6918]]], device='mps:0',
       grad_fn=<AddBackward0>)

### Training

In [ ]:
mse = torch.nn.MSELoss()

In [ ]:
torch.manual_seed(train_seed)
nn_tf_enc.train()
print(nn_tf_enc(x))
loss = mse(nn_tf_enc(x), x)
print(loss.item())
loss.backward()
nn_tf_enc(x)


tensor([[[ 0.6852, -0.1630,  2.2074, -0.9302],
         [ 0.9349,  0.6778,  1.9027, -0.1390],
         [ 0.8390,  0.4592,  1.5378,  0.0823]],

        [[-0.1834,  0.3312,  0.7525,  2.1237],
         [ 0.7906,  0.1192,  1.3922,  0.4578],
         [ 0.8707,  0.1922,  0.5967,  0.7011]]], device='mps:0',
       grad_fn=<AddBackward0>)
0.6275849342346191


tensor([[[ 0.2741, -0.4894,  1.8790, -1.0957],
         [ 0.7022,  0.7676,  0.8320, -0.2268],
         [ 0.7146,  0.2815,  0.8392,  1.0298]],

        [[ 0.5085,  0.0161,  1.4960,  2.1186],
         [ 0.4879,  0.0038,  1.0066,  1.0014],
         [ 1.1819,  0.7470,  0.2558,  0.4909]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [ ]:
torch.manual_seed(train_seed)
tf_enc.train()
print(tf_enc(x))
loss = mse(tf_enc(x), x)
print(loss.item())
loss.backward()
tf_enc(x)

tensor([[[ 0.6852, -0.1630,  2.2074, -0.9302],
         [ 0.9349,  0.6778,  1.9027, -0.1390],
         [ 0.8390,  0.4592,  1.5378,  0.0823]],

        [[-0.1834,  0.3312,  0.7525,  2.1237],
         [ 0.7906,  0.1192,  1.3922,  0.4578],
         [ 0.8707,  0.1922,  0.5967,  0.7011]]], device='mps:0',
       grad_fn=<AddBackward0>)
0.6275849342346191


tensor([[[ 0.2741, -0.4894,  1.8790, -1.0957],
         [ 0.7022,  0.7676,  0.8320, -0.2268],
         [ 0.7146,  0.2815,  0.8392,  1.0298]],

        [[ 0.5085,  0.0161,  1.4960,  2.1186],
         [ 0.4879,  0.0038,  1.0066,  1.0014],
         [ 1.1819,  0.7470,  0.2558,  0.4909]]], device='mps:0',
       grad_fn=<AddBackward0>)